# 👋 Hello World — pi.dev (`@earendil-works/pi-ai`)

This notebook is the first step in exploring [**pi**](https://github.com/earendil-works/pi) — an open-source, multi-provider LLM API and agent framework. Here we just want to prove the toolchain works end-to-end: import the library, build a `Models` collection, pick a model, and get a single response back.

We use the **`@earendil-works/pi-ai`** package, which is the low-level unified LLM API. Later notebooks will build actual agents on top of it (`@earendil-works/pi-agent-core`).

> **Kernel:** this notebook runs on the **TypeScript (tslab)** kernel. See `../README.md` for setup.

## Prerequisites

You need **one** LLM provider API key exported in the environment *before* you launch Jupyter, e.g.:

```bash
export ANTHROPIC_API_KEY=sk-ant-...   # or OPENAI_API_KEY, or GEMINI_API_KEY
jupyter lab
```

pi resolves the key automatically from the environment — no key ever appears in the notebook.

In [ ]:
// 1. Import the unified API and register every built-in provider.
import { type Context } from "@earendil-works/pi-ai";
import { builtinModels } from "@earendil-works/pi-ai/providers/all";

const models = builtinModels();
console.log("pi-ai loaded ✔  — a Models collection with all built-in providers is ready.");

In [ ]:
// 2. Pick a model based on whichever provider key you have in the environment.
//    Add your own line here if you use a different provider.
const candidates: Array<[string, string, string]> = [
  ["ANTHROPIC_API_KEY", "anthropic", "claude-sonnet-4-5"],
  ["OPENAI_API_KEY",    "openai",    "gpt-4o-mini"],
  ["GEMINI_API_KEY",    "google",    "gemini-2.5-flash"],
];

const picked = candidates.find(([envVar]) => process.env[envVar]);
if (!picked) {
  throw new Error(
    "No provider API key found. Export one (e.g. ANTHROPIC_API_KEY) and restart the kernel."
  );
}

const [envVar, provider, modelId] = picked;
const model = models.getModel(provider, modelId);
if (!model) throw new Error(`Model ${provider}/${modelId} not found in the catalog.`);

console.log(`Using ${provider}/${modelId} (auth via ${envVar}).`);

In [ ]:
// 3. Hello world: send one message and print the reply.
const context: Context = {
  systemPrompt: "You are a friendly assistant. Keep answers to one short sentence.",
  messages: [
    { role: "user", content: "Say hello and confirm you are running via the pi.dev library.", timestamp: Date.now() },
  ],
};

const response = await models.completeSimple(model, context);

for (const block of response.content) {
  if (block.type === "text") console.log("🤖", block.text);
}

console.log(
  `\ntokens: ${response.usage.input} in / ${response.usage.output} out` +
  `  |  cost: $${response.usage.cost.total.toFixed(6)}`
);

## ✅ What just happened

1. `builtinModels()` gave us a `Models` collection with every provider registered.
2. `models.getModel(provider, id)` resolved a concrete model.
3. `models.completeSimple(model, context)` ran one request, resolving auth from your environment, and returned a message with `content` blocks plus token/cost `usage`.

## Next steps to explore

- **Streaming**: swap `completeSimple` for `models.streamSimple(model, context)` and iterate the events (`text_delta`, `toolcall_end`, …).
- **Tools**: define a `Tool` with a TypeBox schema and handle `toolCall` blocks — the foundation of an agent loop.
- **Agents**: install `@earendil-works/pi-agent-core` and use the `Agent` class, which wraps the tool-calling loop for you.